In [2]:
#import argparse
import logging

import awswrangler as wr
import pandas as pd

In [4]:
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [5]:
def read_bronze_table(bucket: str, file_name: str, table_name: str) -> pd.DataFrame:
    """
    Read one Bronze CSV table from S3.

    Args:
        bucket (str): source S3 bucket
        file_name (str): file name in Bronze layer
        table_name (str): table name

    Returns:
        Loaded DataFrame.
    """
    path = f"s3://{bucket}/mexico-public-safety-observatory/outputs/{file_name}"

    logger.info(f"Reading {table_name} from {path}")

    try:
        return wr.s3.read_csv(path)
    except Exception as e:
        logger.error(f"Error reading {table_name}: {e}")
        raise

In [6]:
year_scores = read_bronze_table(
                bucket="itam-anlytics-karla",
                file_name="violence_scores_yearly.csv",
                table_name="violence_scores_yearly",
            )
year_scores

2026-05-21 22:02:45,762 - INFO - Reading violence_scores_yearly from s3://itam-anlytics-karla/mexico-public-safety-observatory/outputs/violence_scores_yearly.csv
2026-05-21 22:02:45,778 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


,entidad_nombre,anio,violence_score
0,Aguascalientes,2015,16.405715
1,Aguascalientes,2016,16.098249
2,Aguascalientes,2017,25.622423
3,Aguascalientes,2018,43.774318
4,Aguascalientes,2019,46.248401
...,...,...,...
347,Zacatecas,2021,100.000000
348,Zacatecas,2022,100.000000
349,Zacatecas,2023,77.748474
350,Zacatecas,2024,60.642596


In [8]:
violence_scores = read_bronze_table(
                bucket="itam-anlytics-karla",
                file_name="violence_scores.csv",
                table_name="violence_scores",
            )
violence_scores

2026-05-21 22:03:09,690 - INFO - Reading violence_scores from s3://itam-anlytics-karla/mexico-public-safety-observatory/outputs/violence_scores.csv
2026-05-21 22:03:09,704 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


,entidad_nombre,violence_score,cluster,nivel
0,Nuevo León,100.000000,2,Crítico
1,Colima,86.683358,2,Crítico
2,Quintana Roo,82.770812,2,Crítico
3,Morelos,82.541975,2,Crítico
4,Zacatecas,79.519716,2,Crítico
5,Baja California,69.298142,0,Alto
6,Sinaloa,63.857385,0,Alto
7,Tabasco,63.768295,0,Alto
8,Chihuahua,62.909854,0,Alto
9,Tamaulipas,62.382713,0,Alto


In [15]:
state_history = year_scores[
    year_scores["entidad_nombre"] == "Chihuahua"
].sort_values("anio", ascending=False)
state_history

,entidad_nombre,anio,violence_score
65,Chihuahua,2025,93.844844
64,Chihuahua,2024,100.000000
63,Chihuahua,2023,94.981548
62,Chihuahua,2022,72.034448
61,Chihuahua,2021,64.559877
60,Chihuahua,2020,54.783772
59,Chihuahua,2019,57.517099
58,Chihuahua,2018,56.018348
57,Chihuahua,2017,40.898631
56,Chihuahua,2016,29.552346


In [16]:
state_history.iloc[0]

entidad_nombre    Chihuahua
anio                   2025
violence_score    93.844844
Name: 65, dtype: object

In [20]:
filtered_year_scores = year_scores[
    (year_scores["entidad_nombre"].isin(["Chihuahua", "Aguascalientes"]))
    & (year_scores["anio"].between(2018, 2023))
]

In [22]:
filtered_year_scores[
    filtered_year_scores["entidad_nombre"] == "Chihuahua"
].sort_values("anio")

,entidad_nombre,anio,violence_score
58,Chihuahua,2018,56.018348
59,Chihuahua,2019,57.517099
60,Chihuahua,2020,54.783772
61,Chihuahua,2021,64.559877
62,Chihuahua,2022,72.034448
63,Chihuahua,2023,94.981548


In [12]:
year_scores["entidad_nombre"].unique()

<ArrowStringArray>
[                 'Aguascalientes',                 'Baja California',
             'Baja California Sur',                        'Campeche',
                         'Chiapas',                       'Chihuahua',
                'Ciudad de México',            'Coahuila de Zaragoza',
                          'Colima',                         'Durango',
                      'Guanajuato',                        'Guerrero',
                         'Hidalgo',                         'Jalisco',
             'Michoacán de Ocampo',                         'Morelos',
                          'México',                         'Nayarit',
                      'Nuevo León',                          'Oaxaca',
                          'Puebla',                       'Querétaro',
                    'Quintana Roo',                 'San Luis Potosí',
                         'Sinaloa',                          'Sonora',
                         'Tabasco',                      '

In [23]:
latest_state_score = violence_scores[
     violence_scores["entidad_nombre"] == "Aguascalientes"
     ].iloc[0]
latest_state_score

entidad_nombre    Aguascalientes
violence_score         35.484074
cluster                        3
nivel                      Medio
Name: 19, dtype: object

In [25]:
state_history = year_scores[
    year_scores["entidad_nombre"] == "Aguascalientes"
].sort_values("anio")
state_history

,entidad_nombre,anio,violence_score
0,Aguascalientes,2015,16.405715
1,Aguascalientes,2016,16.098249
2,Aguascalientes,2017,25.622423
3,Aguascalientes,2018,43.774318
4,Aguascalientes,2019,46.248401
5,Aguascalientes,2020,49.627455
6,Aguascalientes,2021,45.218302
7,Aguascalientes,2022,47.359517
8,Aguascalientes,2023,34.955337
9,Aguascalientes,2024,37.360101
